# PARC2026 — S4 Run A Recipe Freeze Gate

20k本学習の直前にだけ使う最終Gateです。

`run_a_recipe.json` を `READY` にするにはすべて必要です。

- D10: provisional best datasetが決着
- M3: 3-model fair comparison完了 + best/shortlist決着
- G1: generalization / augmentation decision完了
- T3: inverse-data ablation evidence完了
- organizer `libero_combined_20hz`: final recipeを再適用し integrity/leakage PASS

public proxyの結果だけではRun AをREADYにしません。
Run Bは、Run AでTrack3だけ明確に弱く、inverse evidenceがpositiveな場合にのみ候補化します。


In [ ]:
# 0/2 Gather all pre-training evidence
import json
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
DRIVE=Path("/content/drive/MyDrive/parc2026-cache")
paths={"D10":DRIVE/"pi05-top2-tiebreak-v1/provisional_best_dataset_recipe.json","M3":DRIVE/"model-benchmark-v1/model_benchmark_result.json","G1":DRIVE/"generalization-screening-v1/generalization_result.json","T3":DRIVE/"track3-inverse-factory-v1/inverse_ablation_result.json","ORGANIZER":DRIVE/"organizer-final-recipe-v1/organizer_recipe_gate.json"}
docs={}; missing=[]
for k,p in paths.items():
    if p.exists(): docs[k]=json.loads(p.read_text()); print(k,"FOUND",p)
    else: missing.append(k); print(k,"MISSING",p)
if missing: print("\nS4 BLOCKED:",missing)
else: print("\nAll evidence files present; validating contracts...")


In [ ]:
# 1/2 Strict S4 validation and Run A/Run B recipe artifacts
import json, time
OUT=DRIVE/"run-a-freeze-v1"; OUT.mkdir(parents=True,exist_ok=True); blocked=[]
if missing: blocked.extend([f"missing:{x}" for x in missing])
else:
    if docs["D10"].get("status")!="DECIDED": blocked.append("D10_not_decided")
    if docs["M3"].get("status")!="DECIDED": blocked.append("M3_not_decided")
    if docs["G1"].get("status")!="PASS": blocked.append("G1_not_pass")
    if docs["T3"].get("status")!="PASS": blocked.append("T3_not_pass")
    org=docs["ORGANIZER"]
    if org.get("status")!="PASS": blocked.append("organizer_recipe_gate_not_pass")
    if org.get("dataset")!="libero_combined_20hz": blocked.append("organizer_source_not_libero_combined_20hz")
    if int(org.get("exact_leakage_group_count",-1))!=0: blocked.append("organizer_exact_leakage")
    if int(org.get("action_leakage_group_count",-1))!=0: blocked.append("organizer_action_leakage")
if blocked:
    blocked_doc={"status":"BLOCKED","blocked_reasons":blocked,"checked_at_unix":time.time(),"hard_rule":"do_not_start_20k_before_all_pretraining_gates"}
    (OUT/"run_a_gate_status.json").write_text(json.dumps(blocked_doc,indent=2)+"\n"); print(json.dumps(blocked_doc,indent=2)); print("\nNo READY Run A recipe was written.")
else:
    run_a={"status":"READY","role":"common_track1_track2_track3_candidate","model":docs["M3"]["selected_model"],"model_revision":docs["M3"]["selected_model_revision"],"dataset":"libero_combined_20hz","dataset_manifest":docs["ORGANIZER"]["dataset_manifest"],"dataset_manifest_sha256":docs["ORGANIZER"]["dataset_manifest_sha256"],"sampling_policy":docs["ORGANIZER"]["sampling_policy"],"augmentation":docs["G1"]["selected_augmentation"],"training":{"adaptation":"LoRA","optimizer_steps":20000,"save_interval":1000,"suggested_eval_steps":[5000,10000,15000,20000],"lora_rank":docs["M3"]["selected_lora_rank"],"learning_rate":docs["M3"]["selected_learning_rate"],"batch_size":docs["M3"]["selected_batch_size"],"grad_accum":docs["M3"]["selected_grad_accum"],"seed":docs["M3"]["selected_seed"]},"required_post_train_eval":["track1","track2","track3","l4_24gb_latency_vram"]}
    (OUT/"run_a_recipe.json").write_text(json.dumps(run_a,indent=2)+"\n"); print(json.dumps(run_a,indent=2))
    run_b_evidence=docs["T3"].get("run_b_evidence",{})
    if run_b_evidence.get("inverse_screening_positive") is True:
        candidate={"status":"CONDITIONAL","trigger_all":["run_a_track1_track2_are_acceptable","run_a_track3_is_clearly_weaker","inverse_screening_shows_positive_track3_evidence"],"validated_inverse_ratio":run_b_evidence.get("best_inverse_ratio"),"base":"continue_from_selected_run_a_checkpoint"}
        (OUT/"run_b_track3_candidate.json").write_text(json.dumps(candidate,indent=2)+"\n"); print("\nRun B candidate recorded (still conditional).")
    else: print("\nRun B not preregistered: no positive inverse evidence.")
    print("\n=== S4 RUN A FREEZE: READY ===")
